In [20]:
import pandas as pd
import numpy as np

# =========================
# 1. 파일 불러오기
# =========================
df_base = pd.read_excel("기준데이터.xlsx")
df_target = pd.read_excel("비교대상데이터.xlsx")


# =========================
# 2. 공통 함수
# =========================
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def to_float(x):
    if pd.isna(x):
        return np.nan
    s = str(x).replace(",", "").strip()
    if s == "":
        return np.nan
    try:
        return float(s)
    except:
        return np.nan

def normalize_jimok(x):
    x = clean_text(x)
    mapping = {
        "대지": "대"
    }
    return mapping.get(x, x)

def area_flag(base, comp, tol=0.01):
    if pd.notna(base) and pd.notna(comp):
        return 1 if abs(base - comp) > tol else 0
    return 0

def jimok_flag(base, comp):
    if base != "" and comp != "":
        return 1 if base != comp else 0
    return 0

def make_code(area_diff, jimok_diff):
    if area_diff == 0 and jimok_diff == 0:
        return 0
    elif area_diff == 1 and jimok_diff == 0:
        return 1
    elif area_diff == 0 and jimok_diff == 1:
        return 2
    elif area_diff == 1 and jimok_diff == 1:
        return 3
    return 0

def exists_value(x):
    if pd.isna(x):
        return 0
    if str(x).strip() == "":
        return 0
    return 1


# =========================
# 3. 컬럼/값 정리
# =========================
for df in [df_base, df_target]:
    df.columns = [clean_text(c) for c in df.columns]

df_base["소재지"] = df_base["소재지"].apply(clean_text)
df_target["소재지"] = df_target["소재지"].apply(clean_text)

df_base["면적"] = df_base["면적"].apply(to_float)
df_base["지목"] = df_base["지목"].apply(normalize_jimok)

for col in ["고시면적", "토지면적", "출자면적"]:
    if col in df_target.columns:
        df_target[col] = df_target[col].apply(to_float)

for col in ["고시지목", "토지지목", "출자지목"]:
    if col in df_target.columns:
        df_target[col] = df_target[col].apply(normalize_jimok)

if "상태" in df_base.columns:
    df_base["상태"] = df_base["상태"].apply(clean_text)
else:
    df_base["상태"] = ""

if "토지이동사유" in df_base.columns:
    df_base["토지이동사유"] = df_base["토지이동사유"].apply(clean_text)
else:
    df_base["토지이동사유"] = ""


# =========================
# 4. 병합
# =========================
target_cols = [
    "소재지",
    "고시면적", "토지면적", "출자면적",
    "고시지목", "토지지목", "출자지목"
]

target_cols = [c for c in target_cols if c in df_target.columns]

df = df_base.merge(
    df_target[target_cols],
    on="소재지",
    how="left"
)


# =========================
# 5. 고시 비교
# =========================
gosi_df = df[[
    "소재지", "면적", "고시면적", "지목", "고시지목"
]].copy()

gosi_df["면적다름"] = gosi_df.apply(lambda r: area_flag(r["면적"], r["고시면적"]), axis=1)
gosi_df["지목다름"] = gosi_df.apply(lambda r: jimok_flag(r["지목"], r["고시지목"]), axis=1)
gosi_df["기타"] = gosi_df.apply(lambda r: make_code(r["면적다름"], r["지목다름"]), axis=1)

gosi_df = gosi_df[[
    "소재지", "면적", "고시면적", "지목", "고시지목", "기타"
]].sort_values(["기타", "소재지"], ascending=[False, True]).reset_index(drop=True)


# =========================
# 6. 토지 비교
# =========================
toji_df = df[[
    "소재지", "면적", "토지면적", "지목", "토지지목"
]].copy()

toji_df["면적다름"] = toji_df.apply(lambda r: area_flag(r["면적"], r["토지면적"]), axis=1)
toji_df["지목다름"] = toji_df.apply(lambda r: jimok_flag(r["지목"], r["토지지목"]), axis=1)
toji_df["기타"] = toji_df.apply(lambda r: make_code(r["면적다름"], r["지목다름"]), axis=1)

toji_df = toji_df[[
    "소재지", "면적", "토지면적", "지목", "토지지목", "기타"
]].sort_values(["기타", "소재지"], ascending=[False, True]).reset_index(drop=True)


# =========================
# 7. 출자 비교
# =========================
chulja_df = df[[
    "소재지", "면적", "출자면적", "지목", "출자지목"
]].copy()

chulja_df["면적다름"] = chulja_df.apply(lambda r: area_flag(r["면적"], r["출자면적"]), axis=1)
chulja_df["지목다름"] = chulja_df.apply(lambda r: jimok_flag(r["지목"], r["출자지목"]), axis=1)
chulja_df["기타"] = chulja_df.apply(lambda r: make_code(r["면적다름"], r["지목다름"]), axis=1)

chulja_df = chulja_df[[
    "소재지", "면적", "출자면적", "지목", "출자지목", "기타"
]].sort_values(["기타", "소재지"], ascending=[False, True]).reset_index(drop=True)


# =========================
# 8. 소재지점검
# - 기준에는 있는데 고시에는 없는 소재지
# =========================
addr_df = df[[
    "소재지", "상태", "토지이동사유",
    "고시면적", "고시지목",
    "토지면적", "토지지목",
    "출자면적", "출자지목"
]].copy()

addr_df["고시존재"] = addr_df.apply(
    lambda r: 1 if (
        exists_value(r["고시면적"]) == 1 or
        exists_value(r["고시지목"]) == 1
    ) else 0,
    axis=1
)

addr_df["토지존재"] = addr_df.apply(
    lambda r: 1 if (
        exists_value(r["토지면적"]) == 1 or
        exists_value(r["토지지목"]) == 1
    ) else 0,
    axis=1
)

addr_df["출자존재"] = addr_df.apply(
    lambda r: 1 if (
        exists_value(r["출자면적"]) == 1 or
        exists_value(r["출자지목"]) == 1
    ) else 0,
    axis=1
)

addr_df = addr_df[addr_df["고시존재"] == 0].copy()

def check_deleted(row):
    status = row["상태"]
    reason = row["토지이동사유"]
    if ("말소" in status) or ("말소" in reason):
        return 1
    return 0

addr_df["말소여부"] = addr_df.apply(check_deleted, axis=1)

def make_register_status(row):
    places = []
    if row["토지존재"] == 1:
        places.append("토지")
    if row["출자존재"] == 1:
        places.append("출자")

    if len(places) == 0:
        return "없음"
    return ", ".join(places)

addr_df["등재현황"] = addr_df.apply(make_register_status, axis=1)

addr_df = addr_df[[
    "소재지",
    "말소여부",
    "등재현황"
]].sort_values(["말소여부", "등재현황", "소재지"], ascending=[False, True, True]).reset_index(drop=True)


# =========================
# 9. 면적불일치 시트
# - 기타 1, 3 포함
# =========================
area_condition = (
    gosi_df["기타"].isin([1, 3]).values |
    toji_df["기타"].isin([1, 3]).values |
    chulja_df["기타"].isin([1, 3]).values
)

area_mismatch_df = df.loc[area_condition, [
    "소재지", "면적", "고시면적", "토지면적", "출자면적"
]].copy()

area_mismatch_df = area_mismatch_df.drop_duplicates(subset=["소재지"]) \
                                   .sort_values("소재지") \
                                   .reset_index(drop=True)


# =========================
# 10. 지목불일치 시트
# - 기타 2, 3 포함
# =========================
jimok_condition = (
    gosi_df["기타"].isin([2, 3]).values |
    toji_df["기타"].isin([2, 3]).values |
    chulja_df["기타"].isin([2, 3]).values
)

jimok_mismatch_df = df.loc[jimok_condition, [
    "소재지", "지목", "고시지목", "토지지목", "출자지목"
]].copy()

jimok_mismatch_df = jimok_mismatch_df.drop_duplicates(subset=["소재지"]) \
                                     .sort_values("소재지") \
                                     .reset_index(drop=True)


# =========================
# 11. 저장
# =========================
output_file = "정합성검정결과.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    gosi_df.to_excel(writer, sheet_name="01_고시비교", index=False)
    toji_df.to_excel(writer, sheet_name="02_토지비교", index=False)
    chulja_df.to_excel(writer, sheet_name="03_출자비교", index=False)
    addr_df.to_excel(writer, sheet_name="04_소재지점검", index=False)
    area_mismatch_df.to_excel(writer, sheet_name="05_면적불일치", index=False)
    jimok_mismatch_df.to_excel(writer, sheet_name="06_지목불일치", index=False)


# =========================
# 12. 출력
# =========================
print("완료")
print("저장 파일:", output_file)

print("\n고시비교 기타값 분포")
print(gosi_df["기타"].value_counts().sort_index())

print("\n토지비교 기타값 분포")
print(toji_df["기타"].value_counts().sort_index())

print("\n출자비교 기타값 분포")
print(chulja_df["기타"].value_counts().sort_index())

print("\n소재지점검 결과")
print("고시에 없는 소재지 수:", len(addr_df))
print("말소 수:", (addr_df["말소여부"] == 1).sum())
print("토지 등재 수:", (addr_df["등재현황"] == "토지").sum())
print("출자 등재 수:", (addr_df["등재현황"] == "출자").sum())
print("토지, 출자 동시 등재 수:", (addr_df["등재현황"] == "토지, 출자").sum())
print("등재 없음 수:", (addr_df["등재현황"] == "없음").sum())

print("\n05_면적불일치 행 수:", len(area_mismatch_df))
print("06_지목불일치 행 수:", len(jimok_mismatch_df))

완료
저장 파일: 정합성검정결과.xlsx

고시비교 기타값 분포
기타
0    365
1     17
2     11
Name: count, dtype: int64

토지비교 기타값 분포
기타
0    362
1     24
2      6
3      1
Name: count, dtype: int64

출자비교 기타값 분포
기타
0    327
1     52
2      9
3      5
Name: count, dtype: int64

소재지점검 결과
고시에 없는 소재지 수: 35
말소 수: 17
토지 등재 수: 7
출자 등재 수: 28
토지, 출자 동시 등재 수: 0
등재 없음 수: 0

05_면적불일치 행 수: 64
06_지목불일치 행 수: 14
